# Week 4 — Prototype dense retrieval (SBERT + FAISS)

Smallest possible end-to-end check: a 6-passage toy corpus, `all-MiniLM-L6-v2` as the encoder, three hand-written queries, and a side-by-side BM25 comparison on the *same* tiny corpus. Numbers here are diagnostic only — they verify the pipeline runs, they are **not** benchmark numbers.

The honest baseline lives in `experiments/run_dense_retrieval.py` (50k qrels-anchored sample of MS MARCO Passage); see `outputs/week04_dense/metrics.json` and `reports/generated/week04_dense.md` for the real numbers.

## Setup

In [1]:
import sys
from pathlib import Path


def _find_project_root(markers=(".git", "pyproject.toml", "requirements.txt")):
    p = Path.cwd().resolve()
    for parent in (p, *p.parents):
        if any((parent / m).exists() for m in markers):
            return parent
    return p


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# macOS libomp workaround: faiss-cpu and torch each ship libomp.dylib.
# Loading both in the same process aborts/segfaults unless we tell the
# runtime it's fine. Must be set before the first faiss/torch import.
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /Users/content/msmarco-genqa


In [2]:
from src.retrieval.dense import DenseRetriever
from src.retrieval.bm25 import BM25Retriever

## 1. Toy corpus

Six passages, deliberately chosen so a sparse retriever can win on lexical overlap and a dense retriever can win on paraphrase / synonym overlap.

In [3]:
corpus = [
    "Canberra is the capital city of Australia.",                              # d0
    "The Eiffel Tower is a wrought-iron lattice tower located in Paris.",      # d1
    "William Shakespeare wrote the tragedy Hamlet in the early 17th century.", # d2
    "Sydney is the largest city in Australia but it is not the capital.",      # d3
    "The Pacific Ocean is the largest ocean on Earth.",                        # d4
    "Photosynthesis converts sunlight into chemical energy in green plants.",  # d5
]
doc_ids = [f"d{i}" for i in range(len(corpus))]

## 2. Build the dense (SBERT + FAISS) and BM25 indexes on the same corpus

In [4]:
dense = DenseRetriever(model_name="sentence-transformers/all-MiniLM-L6-v2", device="cpu")
dense.build(corpus, doc_ids)

bm25 = BM25Retriever(corpus_texts=corpus, doc_ids=doc_ids, k1=1.5, b=0.75)
bm25.build()
"both retrievers ready"

/Users/content/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/6 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/6 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/6 [00:00<?, ?it/s]

'both retrievers ready'

## 3. Three queries — lexical-easy, paraphrase, semantic

We expect both retrievers to pick the obvious answer on lexical queries. The interesting case is Q3, where the query word *"playwright"* doesn't appear in any passage but is semantically close to *"wrote a tragedy"* in d2.

In [5]:
queries = [
    "what is the capital of australia",
    "where is the eiffel tower",
    "which playwright wrote hamlet",
]

def show(label, scores, ids):
    print(f"  {label:6s} top-3: " + ", ".join(f"{d}({s:.3f})" for d, s in zip(ids[:3], scores[:3])))

for q in queries:
    print(f"Q: {q}")
    d_scores, d_ids = dense.retrieve(q, k=3)
    b_scores, b_ids = bm25.retrieve(q, k=3)
    show("dense",  d_scores, d_ids)
    show("bm25",   b_scores, b_ids)
    print()

Q: what is the capital of australia


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  dense  top-3: d0(0.811), d3(0.690), d2(0.167)
  bm25   top-3: d0(0.978), d3(0.900), d1(0.000)

Q: where is the eiffel tower


...

  dense  top-3: d1(0.841), d5(0.069), d3(0.064)
  bm25   top-3: d1(1.347), d5(0.000), d3(0.000)

Q: which playwright wrote hamlet


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  dense  top-3: d2(0.670), d0(0.080), d3(0.068)
  bm25   top-3: d2(1.087), d1(0.000), d3(0.000)



## 4. Where dense vs BM25 actually disagree

On the tiny corpus both retrievers get the right top-1 because Q3 still contains the literal word *"hamlet"*. To see the gap, drop the keyword and force a purely semantic query:

In [6]:
q = "a famous english tragedy from the renaissance"
scores_d, ids_d = dense.retrieve(q, k=3)
scores_b, ids_b = bm25.retrieve(q, k=3)
print("dense:", list(zip(ids_d, [f'{s:.3f}' for s in scores_d])))
print("bm25: ", list(zip(ids_b, [f'{s:.3f}' for s in scores_b])))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

dense: [('d2', '0.585'), ('d4', '0.045'), ('d0', '0.038')]
bm25:  [('d2', '0.543'), ('d1', '0.000'), ('d3', '0.000')]


BM25 has no signal (no content words overlap with the corpus), so it returns near-zero scores in arbitrary order. The dense retriever still ranks d2 (Shakespeare/Hamlet) at the top because the SBERT embedding maps "english tragedy" close to "Shakespeare's tragedy Hamlet". This is the qualitative reason to add dense retrieval in the first place.

## Limitations

- 6-passage corpus is a smoke test, not an evaluation.
- `all-MiniLM-L6-v2` is generic-purpose, not MS-MARCO-tuned.
- FAISS `IndexFlatIP` on 6 vectors is microseconds; latency is only meaningful at the 50k+ scale.

## Next — official dense baseline

Run the script-based pipeline on the 50k qrels-anchored sample:

```bash
python experiments/run_dense_retrieval.py --sample-size 50000 --rebuild-index
python -m src.reporting.build_report --week week04
```
